# Étape 2 : Évaluation des Règles et Création de la Vérité Terrain
Dans cette étape, nous allons confronter nos algorithmes à base de règles (Regex basique vs. Regex contextuelle) à une annotation humaine. L'objectif est de constituer un échantillon robuste et équilibré qui servira de vérité terrain (`label`) pour l'entraînement de nos futurs modèles de Machine Learning.

In [1]:
import pandas as pd
from rules import regle_de_base, regle_avancee_contexte

In [2]:
df = pd.read_csv("./data/df_preproc.csv", sep=",")

## 2.1 Test préliminaire des algorithmes sur un petit échantillon
Nous sélectionnons aléatoirement 10 comptes rendus mentionnant le tabac pour observer le comportement de nos deux approches par règles (`regle_de_base` et `regle_avancee_contexte`).

In [3]:
echantillon_test_regles = df[df['mention_tabac'] == True].sample(10, random_state=42)

echantillon_test_regles["pred_regle_base"] = echantillon_test_regles["text"].apply(regle_de_base)
echantillon_test_regles["pred_regle_avancee"] = echantillon_test_regles["text"].apply(regle_avancee_contexte)

In [4]:
echantillon_test_regles[["id", "pred_regle_base", "pred_regle_avancee"]]

,id,pred_regle_base,pred_regle_avancee
4018,MEDECINE-GERIATRIQUE-00171,Non-fumeur,Non-fumeur
3495,HEPATO-GASTRO-ENTERO-00081,Fumeur actuel,Non-fumeur
3901,MEDECINE-GERIATRIQUE-00014,Ancien fumeur,Ancien fumeur
3035,GYNECOLOGIE-00207_1,Fumeur actuel,Non-fumeur
1544,CHIR-ORTHO-ET-TRAUMATO-00267_1,Fumeur actuel,Non-fumeur
3991,MEDECINE-GERIATRIQUE-00137,Ancien fumeur,Ancien fumeur
3898,MEDECINE-GERIATRIQUE-00011,Ancien fumeur,Ancien fumeur
5441,PNEUMOLOGIE-00388,Non mentionné,Non-fumeur
5651,REANIMATION-00015,Ancien fumeur,Fumeur actuel
4128,MEDECINE-INTERNE-00003,Fumeur actuel,Non-fumeur


In [5]:
for index, row in echantillon_test_regles.iterrows():
    print(f'id : {index}')
    print(f"text : {row.text}\n")

id : 4018
text : Compte rendu d hospitalisation Service de Médecine gériatrique Monsieur Christophe Giner Correas âgé de 77 ans a été hospitalisé en service de Médecine Gériatrique du 03 11 2025 au 17 11 2025 Motif d hospitalisation Admis depuis les urgences pour majoration de dyspnée oedèmes des membres inférieurs et altération de l état général évoluant depuis plusieurs jours Antécédents médicaux chirurgicaux familiaux allergies Antécédents médicaux • Hypertension artérielle ancienne • Fibrillation auriculaire paroxystique suivie en cardiologie • Insuffisance cardiaque chronique à FEVG modérée • Maladie rénale chronique stade 3a • Syndrome parkinsonien idiopathique diagnostiqué il y a 4 ans • Trouble cognitif majeur d allure Alzheimer • Diabète de type 2 traité par insuline • Hypotension orthostatique intermittente • Carence en vitamine B9 découverte en 2023 Antécédents chirurgicaux • Cholécystectomie laparoscopique en 1998 • Hernie inguinale droite opérée en 2015 Allergies connues •

## 2.2 Annotation manuelle (Humain vs Machine)
Après analyse manuelle des 10 textes ci-dessus, nous renseignons la véritable classe dans la colonne `ma_pred`. Cela met en évidence les limites de l'approche par règles (ex: fausses détections sur des conseils de prévention ou des antécédents familiaux).

In [6]:
echantillon_test_regles.loc[4018, "ma_pred"] = "Non-fumeur" # Toxiques non fumeur
echantillon_test_regles.loc[3495, "ma_pred"] = "Non-fumeur" # Pas d intoxication tabagique
echantillon_test_regles.loc[3901, "ma_pred"] = "Ancien fumeur" # Sevrage tabagique ancien 30 ans
echantillon_test_regles.loc[3035, "ma_pred"] = "Non-fumeur" # Pas de tabagisme
echantillon_test_regles.loc[1544, "ma_pred"] = "Non-fumeur" # Non tabagique
echantillon_test_regles.loc[3991, "ma_pred"] = "Ancien fumeur" # Tabac sevré depuis 1975
echantillon_test_regles.loc[3898, "ma_pred"] = "Ancien fumeur" # Tabac sevré depuis 30 ans
echantillon_test_regles.loc[5441, "ma_pred"] = "Non-fumeur" # Pas de consommation de tabac
echantillon_test_regles.loc[5651, "ma_pred"] = "Ancien fumeur" # Tabagisme sevré à 40 paquets année
echantillon_test_regles.loc[4128, "ma_pred"] = "Non-fumeur" # Pas de tabagisme

In [7]:
echantillon_test_regles[["id", "pred_regle_base", "pred_regle_avancee", "ma_pred"]]

,id,pred_regle_base,pred_regle_avancee,ma_pred
4018,MEDECINE-GERIATRIQUE-00171,Non-fumeur,Non-fumeur,Non-fumeur
3495,HEPATO-GASTRO-ENTERO-00081,Fumeur actuel,Non-fumeur,Non-fumeur
3901,MEDECINE-GERIATRIQUE-00014,Ancien fumeur,Ancien fumeur,Ancien fumeur
3035,GYNECOLOGIE-00207_1,Fumeur actuel,Non-fumeur,Non-fumeur
1544,CHIR-ORTHO-ET-TRAUMATO-00267_1,Fumeur actuel,Non-fumeur,Non-fumeur
3991,MEDECINE-GERIATRIQUE-00137,Ancien fumeur,Ancien fumeur,Ancien fumeur
3898,MEDECINE-GERIATRIQUE-00011,Ancien fumeur,Ancien fumeur,Ancien fumeur
5441,PNEUMOLOGIE-00388,Non mentionné,Non-fumeur,Non-fumeur
5651,REANIMATION-00015,Ancien fumeur,Fumeur actuel,Ancien fumeur
4128,MEDECINE-INTERNE-00003,Fumeur actuel,Non-fumeur,Non-fumeur


## 2.3 Construction de l'échantillon
Nous générons un échantillon équilibré de 1200 documents au total (600 positifs, 600 négatifs). Cet échantillon sera exporté pour être annoté.

In [8]:
id_annote = [] + echantillon_test_regles["id"].to_list()
df_filtre = df[~df['id'].isin(id_annote)]

# 60 documents avec mention du tabac et 60 autres sans mention
echantillon_positif = df_filtre[df_filtre['mention_tabac'] == True].sample(60, random_state=42)
echantillon_negatif = df_filtre[df_filtre['mention_tabac'] == False].sample(60, random_state=42)

echantillon = pd.concat([echantillon_positif, echantillon_negatif]).sample(frac=1, random_state=42)

del df_filtre
del echantillon_positif
del echantillon_negatif

In [9]:
echantillon["pred_regle_base"] = echantillon["text"].apply(regle_de_base)
echantillon["pred_regle_avancee"] = echantillon["text"].apply(regle_avancee_contexte)
echantillon["ma_pred"] = None

In [10]:
for index, row in echantillon.iloc[0:10].iterrows():
    print(f'id : {index}')
    print(f"text : {row.text}\n")

id : 5241
text : Compte rendu d hospitalisation du service de Pneumologie Monsieur Michel Bouzellatat 81 ans est hospitalisé dans le service de Pneumologie du 03 02 2021 au 09 02 2021 Motif d hospitalisation Insuffisance respiratoire aiguë sur chronique Antécédents médicaux chirurgicaux familiaux allergies Obésité Syndrome de l apnée du sommeil BPCO post tabagique avec emphysème pulmonaire EFR 2016 VEMS 65 PaCO2 33 OLD 1 L min à l effort Hypertrophie bénigne de prostate Insuffisance rénale chronique DFG de base non connu Anévrysme de l aorte abdominale opéré avec pose d endoprothèse Hypertension artérielle essentielle Sténose urètre et col vessie opérée 2021 Pose d endoprothèse aortique abdominale 2014 Prothèse épaule gauche Mode de vie Vit avec son fils Autonome pour tous les actes de la vie quotidienne sort de chez lui prend le bus déjeune au restaurant OLD de temps en temps pas de trouble cognitif Tabagisme sevré à 35 PA Pas d intoxication alcoolique Traitement à l entrée BISOPROLOL

In [11]:
echantillon.iloc[0, echantillon.columns.get_loc("ma_pred")] = "Fumeur actuel" 
echantillon.iloc[1, echantillon.columns.get_loc("ma_pred")] = "Ancien fumeur" 
echantillon.iloc[2, echantillon.columns.get_loc("ma_pred")] = "Non mentionné" 
echantillon.iloc[3, echantillon.columns.get_loc("ma_pred")] = "Ancien fumeur" 
echantillon.iloc[4, echantillon.columns.get_loc("ma_pred")] = "Non mentionné"
echantillon.iloc[5, echantillon.columns.get_loc("ma_pred")] = "Non mentionné" 
echantillon.iloc[6, echantillon.columns.get_loc("ma_pred")] = "Non mentionné" 
echantillon.iloc[7, echantillon.columns.get_loc("ma_pred")] = "Fumeur actuel" 
echantillon.iloc[8, echantillon.columns.get_loc("ma_pred")] = "Ancien fumeur" 
echantillon.iloc[9, echantillon.columns.get_loc("ma_pred")] = "Non mentionné" 

In [12]:
echantillon.iloc[0:10][["id", "pred_regle_base", "pred_regle_avancee", "ma_pred"]]

,id,pred_regle_base,pred_regle_avancee,ma_pred
5241,PNEUMOLOGIE-00135,Ancien fumeur,Fumeur actuel,Fumeur actuel
4269,MEDECINE-INTERNE-00197,Fumeur actuel,Non-fumeur,Ancien fumeur
3876,MALADIES-INFECTIEUSES-00838,Ancien fumeur,Ancien fumeur,Non mentionné
698,CANCERO-ADULTE-00320,Ancien fumeur,Ancien fumeur,Ancien fumeur
1101,CHIR-ORTHO-ET-TRAUMATO-00030_3,Fumeur actuel,Fumeur actuel,Non mentionné
743,CANCERO-ADULTE-00401,Non mentionné,Non mentionné,Non mentionné
2591,CHIRURGIE-VISCERALE-00419_1,Non mentionné,Non mentionné,Non mentionné
4581,NEUROLOGIE-00042,Non mentionné,Non-fumeur,Fumeur actuel
561,CANCERO-ADULTE-00134,Fumeur actuel,Non-fumeur,Ancien fumeur
304,ANATOMOPATHOLOGIE-00365,Non mentionné,Non mentionné,Non mentionné


In [13]:
echantillon.to_csv('./data/echantillon_a_annote_bis.csv', sep=',')

In [14]:
id_annote = [] + echantillon_test_regles["id"].to_list() + echantillon["id"].to_list()
df_filtre = df[~df['id'].isin(id_annote)]

# 540 documents avec mention du tabac et 540 autres sans mention
echantillon_positif = df_filtre[df_filtre['mention_tabac'] == True].sample(540, random_state=42)
echantillon_negatif = df_filtre[df_filtre['mention_tabac'] == False].sample(540, random_state=42)

echantillon_1 = pd.concat([echantillon_positif, echantillon_negatif]).sample(frac=1, random_state=42)

echantillon_1["pred_regle_base"] = echantillon_1["text"].apply(regle_de_base)
echantillon_1["pred_regle_avancee"] = echantillon_1["text"].apply(regle_avancee_contexte)
echantillon_1["ma_pred"] = None

del df_filtre
del echantillon_positif
del echantillon_negatif

In [15]:
echantillon_1.to_csv('./data/echantillon_1_a_annote_bis.csv', sep=',')